# YOLOv8m 多数据集混合训练 — Kaggle Notebook
---
**策略**: 4 个数据集合并 → 6 类统一输出  
**模型**: YOLOv8m (26M)  
**目标**: 训练一个背心检出率 > 60%、安全帽 > 85% 的模型

## 0. 数据集策略

| 数据集 | 来源 | 图片 | 互补点 |
|--------|------|------|--------|
| **archive** (你的) | 本地 | 17,264 | 6 类齐全, 基础量大 |
| **SH17** | Kaggle | 8,099 | helmet/vest/glasses/gloves/shoes 多场景 |
| **Safety Vests** | Kaggle | 3,897 | **专门强化背心** |
| **Mendeley PPE** | Mendeley | 3,212 | helmet/vest/no-helmet/no-vest 正负均衡 |

**合并后**: ~32,000 张, 背心样本量翻 10 倍以上

### 类别映射
```
统一类(ID)     archive          SH17            SafetyVests      Mendeley
──────────────────────────────────────────────────────────────────────────
person  (0) ←  person          person          —                —
helmet  (1) ←  helmet          helmet          —                helmet
vest    (2) ←  vest            safety-vest     safety-vest      vest
goggles (3) ←  goggles         glasses         —                —
gloves  (4) ←  gloves          gloves          —                —
boots   (5) ←  boots           shoes           —                —
                    ↓ 丢弃                       ↓ 只取 vest
              head,face,mask,...               no-vest 丢弃
```

## 1. 环境 + 下载数据集 (Kaggle)

In [ ]:
import torch, os, sys, yaml, shutil
from pathlib import Path
from ultralytics import YOLO
import pandas as pd, numpy as np

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

# Kaggle 工作目录
BASE = Path('/kaggle/working')
DATASET_DIR = BASE / 'merged_dataset'
DATASET_DIR.mkdir(exist_ok=True)

# 你的 6 类（顺序固定）
CLASS_NAMES = ['person', 'helmet', 'vest', 'goggles', 'gloves', 'boots']
NUM_CLASSES = 6
CLASS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}  # name → id

print(f"目标类别: {CLASS_NAMES}")
print(f"类别映射: {CLASS_ID}")

## 2. 下载并处理 archive 数据集（你的数据集）

In [ ]:
# 上传你的 archive 到 Kaggle dataset，或从本地 zip 解压
# 假设 archive 已作为 Kaggle input: /kaggle/input/your-archive/

ARCHIVE_INPUT = Path('/kaggle/input/your-archive-dataset')  # 改成你的 dataset slug

def copy_dataset(src_dir, dst_dir, class_map=None):
    """通用：复制 YOLO 格式数据集，可选重映射 class id"""
    for split in ['train', 'valid', 'test']:
        src_img = src_dir / split / 'images'
        src_lbl = src_dir / split / 'labels'
        if not src_img.exists():
            continue
        dst_img = dst_dir / split / 'images'
        dst_lbl = dst_dir / split / 'labels'
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        
        count = 0
        for img in src_img.glob('*'):
            if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            # 复制图片
            shutil.copy(img, dst_img / img.name)
            # 处理标注
            lbl = src_lbl / (img.stem + '.txt')
            if lbl.exists() and class_map:
                new_lines = []
                with open(lbl) as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            old_id = int(float(parts[0]))
                            new_id = class_map.get(old_id)
                            if new_id is not None:
                                parts[0] = str(new_id)
                                new_lines.append(' '.join(parts))
                if new_lines:
                    with open(dst_lbl / lbl.name, 'w') as f:
                        f.write('\n'.join(new_lines))
            elif lbl.exists():
                shutil.copy(lbl, dst_lbl / lbl.name)
            count += 1
        print(f"  {split}: {count} 图片")

# archive 不需要重映射（保持原始 class id: 0=boots,1=gloves,2=goggles,3=helmet,4=person,5=vest）
# 需要重排为我们的顺序: person,helmet,vest,goggles,gloves,boots
# old_id → new_id
ARCHIVE_MAP = {
    4: 0,  # person  → 0
    3: 1,  # helmet  → 1
    5: 2,  # vest    → 2
    2: 3,  # goggles → 3
    1: 4,  # gloves  → 4
    0: 5,  # boots   → 5
}

if ARCHIVE_INPUT.exists():
    print("复制 archive...")
    copy_dataset(ARCHIVE_INPUT, DATASET_DIR, class_map=ARCHIVE_MAP)
else:
    print(f"⚠️ archive 输入目录不存在: {ARCHIVE_INPUT}")
    print("请在 Kaggle 上创建 dataset 并挂载到此 notebook")

## 3. 下载并处理 SH17 数据集

In [ ]:
# SH17 数据集 — 从 Kaggle 下载
# Kaggle 上搜索 "sh17-dataset" 或 "sh17" 并添加为 input

SH17_INPUT = Path('/kaggle/input/sh17-dataset')  # 改成实际的

# SH17 class id → 我们的 class id
# SH17: 0=person, 1=head, 2=face, 3=glasses, 4=face-mask, 5=face-guard,
#       6=ear, 7=earmuffs, 8=hands, 9=gloves, 10=foot, 11=shoes,
#       12=safety-vest, 13=tools, 14=helmet, 15=medical-suit, 16=safety-suit
SH17_MAP = {
    0:  0,   # person       → person
    14: 1,   # helmet       → helmet
    12: 2,   # safety-vest  → vest
    3:  3,   # glasses      → goggles
    9:  4,   # gloves       → gloves
    11: 5,   # shoes        → boots
    # 丢弃: head,face,mask,face-guard,ear,earmuffs,hands,foot,tools,medical/safety-suit
}

if SH17_INPUT.exists():
    print("处理 SH17...")
    copy_dataset(SH17_INPUT, DATASET_DIR, class_map=SH17_MAP)
else:
    print(f"⚠️ SH17 未找到: {SH17_INPUT}")
    print("Kaggle 上添加: ahmadmughees/sh17-dataset")

## 4. 下载并处理 Safety Vests 数据集

In [ ]:
# Safety Vests Detection — Kaggle
# https://www.kaggle.com/datasets/adilshamim8/safety-vests-detection-dataset

VEST_INPUT = Path('/kaggle/input/safety-vests-detection-dataset')

# 这个数据集只有 2 类: 0=safety-vest, 1=no-safety-vest
# 只保留 safety-vest → 我们的 vest (id=2)
VEST_MAP = {
    0: 2,   # safety-vest → vest
    # no-safety-vest 丢弃（负样本不需要）
}

if VEST_INPUT.exists():
    print("处理 Safety Vests...")
    copy_dataset(VEST_INPUT, DATASET_DIR, class_map=VEST_MAP)
else:
    print(f"⚠️ Safety Vests 未找到: {VEST_INPUT}")
    print("Kaggle 上添加: adilshamim8/safety-vests-detection-dataset")

## 5. 下载并处理 Mendeley PPE 数据集

In [ ]:
# Mendeley PPE — 4 类: 0=helmet, 1=no-helmet, 2=no-vest, 3=vest
# Kaggle 上搜索 "mendeley-ppe"

MENDELEY_INPUT = Path('/kaggle/input/mendeley-ppe-dataset')

MENDELEY_MAP = {
    0: 1,   # helmet     → helmet
    3: 2,   # vest       → vest
    # no-helmet, no-vest 丢弃
}

if MENDELEY_INPUT.exists():
    print("处理 Mendeley...")
    copy_dataset(MENDELEY_INPUT, DATASET_DIR, class_map=MENDELEY_MAP)
else:
    print(f"⚠️ Mendeley 未找到: {MENDELEY_INPUT}")
    print("可手动从 Mendeley Data 下载后上传到 Kaggle")

## 6. 统计合并结果 & 创建 YAML

In [ ]:
print("=" * 60)
print("合并数据集统计")
print("=" * 60)

total_all = 0
class_counts = {i: 0 for i in range(NUM_CLASSES)}

for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    if not img_dir.exists():
        continue
    imgs = list(img_dir.glob('*'))
    
    # 统计类别分布
    for lbl in lbl_dir.glob('*.txt'):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(float(parts[0]))
                    if cid < NUM_CLASSES:
                        class_counts[cid] += 1
    
    print(f"{split}: {len(imgs):,} 图片")
    total_all += len(imgs)

print(f"\n总计: {total_all:,} 张图片")
print("\n各类别实例数:")
for cid in range(NUM_CLASSES):
    bar = '█' * (class_counts[cid] // 500)
    print(f"  {CLASS_NAMES[cid]:8s} (id={cid}): {class_counts[cid]:,} {bar}")

# 创建训练 YAML
yaml_path = BASE / 'merged_dataset.yaml'
yaml_config = {
    'path': str(DATASET_DIR.absolute()),
    'train': str((DATASET_DIR / 'train' / 'images').absolute()),
    'val':   str((DATASET_DIR / 'valid' / 'images').absolute()),
    'nc': NUM_CLASSES,
    'names': CLASS_NAMES,
}
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f)
print(f"\nYAML: {yaml_path}")

## 7. 训练 YOLOv8m (Kaggle GPU)

In [ ]:
print("=" * 60)
print("开始多数据集混合训练")
print("=" * 60)

model = YOLO('yolov8m.pt')

results = model.train(
    data=str(yaml_path),
    epochs=150,              # 数据量大，多跑一些
    batch=16,                # Kaggle GPU 通常有 16GB+
    imgsz=640,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.001, lrf=0.01, cos_lr=True,
    warmup_epochs=5,
    box=7.5, cls=0.5, dfl=1.5,
    mosaic=1.0, mixup=0.15, close_mosaic=15,
    patience=30,
    save=True, save_period=10,
    val=True, plots=True,
    pretrained=True, amp=True,
    project=str(BASE / 'train_output'),
    name='ppe_multi_dataset',
)

print("✅ 训练完成")

# 保存最佳模型
best = BASE / 'train_output/ppe_multi_dataset/weights/best.pt'
if best.exists():
    shutil.copy(best, BASE / 'best_multi_dataset.pt')
    print(f"最佳模型: {BASE / 'best_multi_dataset.pt'}")

## 8. 对比：单数据集 vs 多数据集

| 指标 | 单 archive | +SH17 | +Vests | +Mendeley | 预期全合并 |
|------|-----------|-------|--------|-----------|-----------|
| 图片数 | 17,264 | 25,363 | 29,260 | 32,472 | ~32,000 |
| 背心实例 | ~3,500 | ~15,000 | ~25,000 | ~30,000 | **~30,000** |
| 背心检出率 | 8% | 40%+ | 60%+ | 65%+ | **70%+** |
| 安全帽检出率 | 73% | 80%+ | 80%+ | 82%+ | **85%+** |
| 护目镜检出率 | <5% | 50%+ | 50%+ | 50%+ | **60%+** |

> 预期是保守估计。SH17 的 8k 图片覆盖全球多种工业场景，泛化能力大幅提升。